# VAYU Climate Digital Twin — Kaggle GPU Training

**Accelerator**: GPU T4 × 1 (16 GB) or P100 (16 GB)  
**Target**: Train VayuClimateModel on IMD 2010-2024, validate 2021-2023, test 2024  
**Dataset**: Upload `data/processed/` directory as Kaggle Dataset named `vayu-imd-processed`

## Setup
1. Enable GPU: Settings → Accelerator → GPU T4 x1
2. Add Dataset: `vayu-imd-processed` (your uploaded processed NetCDF files)
3. Run all cells top to bottom

In [ ]:
# ── Environment check ─────────────────────────────────────────────────────────
import subprocess, sys, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — switch accelerator to GPU in Settings!')
print('Python:', sys.version)

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
!pip install -q torch-geometric imdlib xarray scipy netCDF4 tenacity structlog
print('Dependencies installed')

In [ ]:
# ── Mount project code ─────────────────────────────────────────────────────────
import sys, os

# Kaggle dataset path — adjust if your dataset has a different name
DATA_DIR = '/kaggle/input/vayu-imd-processed'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Clone/copy the project source (or add as another dataset)
if not os.path.exists('/kaggle/working/vayu'):
    !git clone https://github.com/YOUR_USERNAME/vayu-climate-digital-twin.git /kaggle/working/vayu

sys.path.insert(0, '/kaggle/working/vayu')
print('Project path added')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import logging
import torch
import xarray as xr
import numpy as np
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s'
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Model config for Kaggle GPU ────────────────────────────────────────────────
from ai_engine.config import ModelConfig

config = ModelConfig(
    # Keep within Kaggle T4 VRAM (16 GB)
    gnn_hidden_dim=128,
    gnn_num_layers=3,
    transformer_d_model=256,
    transformer_nhead=8,
    transformer_num_layers=4,
    transformer_dim_feedforward=512,
    # Training
    max_epochs=100,
    batch_size=4,       # small batch to fit GPU; gradient accumulation below
    learning_rate=1e-4,
    weight_decay=1e-5,
)
print(f'num_nodes={config.num_nodes}, forecast_horizon={config.forecast_horizon}')

In [ ]:
# ── Load processed sequences ──────────────────────────────────────────────────
from data_ingestion.graph_builder import ClimateGraphBuilder

data_dir = Path(DATA_DIR)
train_pt = data_dir / 'train_sequences.pt'
val_pt   = data_dir / 'val_sequences.pt'

if train_pt.exists() and val_pt.exists():
    train_sequences = torch.load(str(train_pt), map_location='cpu')
    val_sequences   = torch.load(str(val_pt),   map_location='cpu')
    print(f'Loaded from disk: {len(train_sequences)} train, {len(val_sequences)} val')
else:
    # Fall back: build from NetCDF files
    print('Building sequences from NetCDF files...')
    builder = ClimateGraphBuilder()

    nc_files = list(data_dir.glob('normalized_*.nc'))
    if not nc_files:
        raise FileNotFoundError(f'No normalized_*.nc files in {data_dir}')

    ds = xr.open_mfdataset(nc_files, combine='by_coords')
    print(f'Dataset: {ds.dims}')

    # Temporal split 2010-2020 train, 2021-2023 val, 2024 test
    train_ds = ds.sel(time=slice('2010', '2020'))
    val_ds   = ds.sel(time=slice('2021', '2023'))

    train_sequences = builder.create_training_sequences(
        train_ds, input_window=config.input_window, target_window=config.forecast_horizon
    )
    val_sequences = builder.create_training_sequences(
        val_ds, input_window=config.input_window, target_window=config.forecast_horizon
    )

    # Save for reuse
    torch.save(train_sequences, '/kaggle/working/train_sequences.pt')
    torch.save(val_sequences,   '/kaggle/working/val_sequences.pt')
    print(f'Built: {len(train_sequences)} train, {len(val_sequences)} val')

In [ ]:
# ── Build model + loss ────────────────────────────────────────────────────────
from ai_engine.climate_model import VayuClimateModel
from ai_engine.loss_functions import PhysicsInformedLoss

model = VayuClimateModel(config)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params/1e6:.2f}M')

loss_fn = PhysicsInformedLoss(
    lambda_conservation=config.lambda_conservation,
    lambda_smoothness=config.lambda_smoothness,
).to(device)

In [ ]:
# ── Training with gradient accumulation ───────────────────────────────────────
# Gradient accumulation lets us simulate batch_size=16 on a GPU with batch_size=4
from ai_engine.trainer import VayuTrainer

trainer = VayuTrainer(
    model=model,
    loss_fn=loss_fn,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
)

print('Starting training...')
history = trainer.train(
    train_sequences=train_sequences,
    val_sequences=val_sequences,
    config=config,
    early_stopping_patience=15,
)

print('Training complete!')
print(f'Best val_loss: {min(history["val_loss"]):.4f}')

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['epochs'], history['train_loss'], label='Train Loss')
axes[0].plot(history['epochs'], history['val_loss'],   label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('VAYU Training Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['epochs'], history['val_r2'], label='Val R² (Tmax)', color='green')
axes[1].axhline(0.85, color='red', linestyle='--', label='Target R²=0.85')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('R²')
axes[1].set_title('Validation R² Score')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')

In [ ]:
# ── Test set evaluation ───────────────────────────────────────────────────────
test_pt = data_dir / 'test_sequences.pt'

if test_pt.exists():
    test_sequences = torch.load(str(test_pt), map_location='cpu')
    test_results = trainer.evaluate_test_set(test_sequences)

    print('\n=== Test Set Results (2024-2025) ===')
    for var, metrics in test_results.items():
        print(f'  {var:12s}: R²={metrics["r2"]:.3f}, RMSE={metrics["rmse"]:.3f}, '
              f'MAE={metrics["mae"]:.3f}, Skill={metrics["skill_score"]:.3f}')

    # Check against Kiro requirements
    tmax_r2 = test_results.get('temp_max', {}).get('r2', 0)
    tmin_r2 = test_results.get('temp_min', {}).get('r2', 0)
    rain_r2 = test_results.get('rainfall', {}).get('r2', 0)

    print(f'\nRequirement check:')
    print(f'  Tmax R²≥0.85: {tmax_r2:.3f} → {"PASS" if tmax_r2 >= 0.85 else "FAIL"}')
    print(f'  Tmin R²≥0.85: {tmin_r2:.3f} → {"PASS" if tmin_r2 >= 0.85 else "FAIL"}')
    print(f'  Rain R²≥0.70: {rain_r2:.3f} → {"PASS" if rain_r2 >= 0.70 else "FAIL"}')
else:
    print('No test sequences found — skipping test evaluation')

In [ ]:
# ── Save checkpoint for AWS deployment ───────────────────────────────────────
import shutil

best_ckpt = Path(CHECKPOINT_DIR) / 'vayu_best.pt'
if best_ckpt.exists():
    shutil.copy(best_ckpt, '/kaggle/working/vayu_best.pt')
    size_mb = best_ckpt.stat().st_size / 1e6
    print(f'Checkpoint saved: /kaggle/working/vayu_best.pt ({size_mb:.1f} MB)')
    print('Download this file and upload to S3: aws s3 cp vayu_best.pt s3://vayu-models/checkpoints/')
else:
    print('No best checkpoint found — training may not have converged')

## Next Steps After Training

1. **Download** `vayu_best.pt` from Kaggle Output
2. **Upload to S3**:
   ```bash
   aws s3 cp vayu_best.pt s3://vayu-climate-models/checkpoints/vayu_best.pt
   ```
3. **Trigger ECS deployment** (CDK will mount S3 checkpoint automatically)
4. **Verify**: `curl https://api.vayu-climate.com/health`

## Kaggle Quota Tips
- Each run ≈ 2-4 hours on T4 (30h/week quota)
- Save checkpoint every 5 epochs to resume if quota runs out
- Use `early_stopping_patience=15` to auto-stop when converged
- Enable Accelerator **T4 x2** for 2× speed if available